In [160]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import pickle
import bs4 as bs
import urllib.request
import nltk
import spacy
import random
import string
import re

In [161]:
base_treinamento = pd.read_csv("tw_treinamento.csv",delimiter=";")
base_treinamento

,id,tweet_text,tweet_date,sentiment,query_used
0,1050785521201541121,@Laranjito76 A pessoa certa para isso seria o ...,Fri Oct 12 16:29:25 +0000 2018,1,:)
1,1050785431955140608,"@behin_d_curtain Para mim, é precisamente o co...",Fri Oct 12 16:29:04 +0000 2018,1,:)
2,1050785401248645120,Vou fazer um video hoje... estou pensando em f...,Fri Oct 12 16:28:56 +0000 2018,1,:)
3,1050785370982547461,"aaaaaaaa amei tanto essas polaroids, nem sei e...",Fri Oct 12 16:28:49 +0000 2018,1,:)
4,1050785368902131713,Valoriza o coração do menininho que vc tem. El...,Fri Oct 12 16:28:49 +0000 2018,1,:)
...,...,...,...,...,...
49995,1046762827053232128,:( é tão lindo que dói https://t.co/GqnpgyWWxB,Mon Oct 01 14:04:40 +0000 2018,0,:(
49996,1046762813362966529,"@veraluciarj Pois é.. tenho problema c/ ""coisa...",Mon Oct 01 14:04:37 +0000 2018,0,:(
49997,1046762806392082432,eu te amo tanto minja vidinha meu bem mais pre...,Mon Oct 01 14:04:35 +0000 2018,0,:(
49998,1046762752071618560,@itsLary @jessboluda Pfvor :(,Mon Oct 01 14:04:22 +0000 2018,0,:(


In [162]:
base_treinamento.drop(["id","tweet_date","query_used"],axis=1,inplace=True)

In [163]:
base_treinamento

,tweet_text,sentiment
0,@Laranjito76 A pessoa certa para isso seria o ...,1
1,"@behin_d_curtain Para mim, é precisamente o co...",1
2,Vou fazer um video hoje... estou pensando em f...,1
3,"aaaaaaaa amei tanto essas polaroids, nem sei e...",1
4,Valoriza o coração do menininho que vc tem. El...,1
...,...,...
49995,:( é tão lindo que dói https://t.co/GqnpgyWWxB,0
49996,"@veraluciarj Pois é.. tenho problema c/ ""coisa...",0
49997,eu te amo tanto minja vidinha meu bem mais pre...,0
49998,@itsLary @jessboluda Pfvor :(,0


In [164]:
from spacy.lang.pt.stop_words import STOP_WORDS
stop_words = STOP_WORDS

In [165]:
pln = spacy.load("pt_core_news_sm")

In [166]:
def preprocessamento(texto):
    texto = texto.lower()
    
    texto = re.sub(r"@[A-Za-z0-9$-_@.&+]+"," ",texto)

    texto = re.sub(r"https?://[A-Za-z0-9./]+"," ",texto)

    texto = re.sub(r" +"," ",texto)

    lista_emocoes = {':)': 'emocaopositiva',
                   ':d': 'emocaopositiva',
                   ':(': 'emocaonegativa'}
    
    for emocoes in lista_emocoes:
        texto = texto.replace(emocoes,lista_emocoes[emocoes])

    documento = pln(texto)

    lista = []
    for token in documento:
        lista.append(token.lemma_)

    lista = [palavra for palavra in lista if palavra not in stop_words and palavra not in  string.punctuation]
    lista = " ".join([str(elemento) for elemento in lista if not elemento.isdigit()])

    return lista

In [167]:
base_treinamento["tweet_text"] = base_treinamento["tweet_text"].apply(preprocessamento)

In [168]:
base_treinamento_final = []

for texto,emocao in zip(base_treinamento["tweet_text"],base_treinamento["sentiment"]):
    if emocao == 1:
        dic = {"POSITIVO":True,"NEGATIVO":False}
    else:
        dic = {"POSITIVO":False,"NEGATIVO":True}

    base_treinamento_final.append([texto,dic.copy()])

In [169]:
len(base_treinamento_final)

50000

In [170]:
modelo = spacy.blank("pt")
textcat = modelo.add_pipe("textcat")
textcat.add_label("POSITIVO")
textcat.add_label("NEGATIVO")
historico = [ ]
from spacy.training import Example

In [171]:
modelo.begin_training()
for epoca in range(5):
    random.shuffle(base_treinamento_final)
    losses = {}
    for batch in spacy.util.minibatch(base_treinamento_final,512):
        textos = [modelo(texto) for texto,entities in batch]
        annotations = [{"cats":entities} for texto,entities in batch]
        examples = [Example.from_dict(doc, annotation) for doc, annotation in zip(
            textos, annotations
        )]
        modelo.update(examples, losses=losses)
        historico.append(losses)
    if epoca % 5 == 0:
        print(losses)
    

{'textcat': 1.7428134632652927}


In [172]:
historico_loss = []
for i in historico:
    historico_loss.append(i.get("textcat"))

In [173]:
historico_loss = np.array(historico_loss)
historico_loss

array([1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281346,
       1.74281346, 1.74281346, 1.74281346, 1.74281346, 1.74281

In [174]:
modelo.to_disk("modelo_tw")

In [175]:
modelo_carregado = spacy.load("modelo_tw")

In [176]:
base_dados_teste = pd.read_csv("tw_teste.csv",delimiter=";")
base_dados_teste

,id,tweet_text,tweet_date,sentiment,query_used
0,1029536486021099522,@Gazo1a Nossa! Muito obrigada :),Wed Aug 15 01:13:20 +0000 2018,1,:)
1,1029536496368406528,@BerzGamer vai pa puta que te pariu :),Wed Aug 15 01:13:23 +0000 2018,1,:)
2,1029536531655131137,QUER MAIS DESCONTOS? (14/08) ⭐⭐⭐⭐⭐ 🌐 Confira n...,Wed Aug 15 01:13:31 +0000 2018,1,:)
3,1029536560117678081,"EU VOU PEGAR VCS, ME AJUDEM GALERA, PELO AMOR ...",Wed Aug 15 01:13:38 +0000 2018,1,:)
4,1029536605852377088,Estávamos em casa do Zé e eu estava a morrer d...,Wed Aug 15 01:13:49 +0000 2018,1,:)
...,...,...,...,...,...
4995,1030528364145201153,@ol_cdanilo parece livro de autoajuda :(,Fri Aug 17 18:54:42 +0000 2018,0,:(
4996,1030528418235015168,@tatazoquita aaaaa sinto muito :((,Fri Aug 17 18:54:55 +0000 2018,0,:(
4997,1030528446122930176,To começando a sentir dor de novo e meu irmão ...,Fri Aug 17 18:55:02 +0000 2018,0,:(
4998,1030528453752352769,@ichbintw parece que no dia que toma a vacina ...,Fri Aug 17 18:55:04 +0000 2018,0,:(


In [177]:
base_dados_teste.drop(["id","tweet_date","query_used"],axis=1,inplace=True)


In [178]:
base_dados_teste

,tweet_text,sentiment
0,@Gazo1a Nossa! Muito obrigada :),1
1,@BerzGamer vai pa puta que te pariu :),1
2,QUER MAIS DESCONTOS? (14/08) ⭐⭐⭐⭐⭐ 🌐 Confira n...,1
3,"EU VOU PEGAR VCS, ME AJUDEM GALERA, PELO AMOR ...",1
4,Estávamos em casa do Zé e eu estava a morrer d...,1
...,...,...
4995,@ol_cdanilo parece livro de autoajuda :(,0
4996,@tatazoquita aaaaa sinto muito :((,0
4997,To começando a sentir dor de novo e meu irmão ...,0
4998,@ichbintw parece que no dia que toma a vacina ...,0


In [179]:
base_dados_teste["tweet_text"] = base_dados_teste["tweet_text"].apply(preprocessamento)

In [180]:
previsoes_teste = []
for texto in base_dados_teste["tweet_text"]:
    previsao = modelo_carregado(texto)
    previsoes_teste.append(previsao.cats)

In [181]:
previsoes_teste

[{'POSITIVO': 0.9999998807907104, 'NEGATIVO': 1.2517597269834368e-07},
 {'POSITIVO': 1.0, 'NEGATIVO': 5.1172818871236814e-08},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 1.4132852754755731e-07},
 {'POSITIVO': 0.9999996423721313, 'NEGATIVO': 4.0420164282295445e-07},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 1.1164279101194552e-07},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 8.246607308137754e-08},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 6.626841297929786e-08},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 6.320348688859667e-08},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 1.3746004867698502e-07},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 1.0185550536334631e-07},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 1.4961513272737648e-07},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 6.148002995587376e-08},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 1.5484717152958183e-07},
 {'POSITIVO': 0.9999998807907104, 'NEGATIVO': 7.980180782851676e-08},
 {'POSITIVO': 0.9999998807

In [182]:
previsao_final = []
for previsao in previsoes_teste:
    if previsao["POSITIVO"] > previsao["NEGATIVO"]:
        previsao_final.append(1)
    else:
        previsao_final.append(0)

previsao_final = np.array(previsao_final)


In [183]:
respostas_reais = base_dados_teste["sentiment"].values

In [184]:
respostas_reais

array([1, 1, 1, ..., 0, 0, 0])

In [185]:
previsao_final

array([1, 1, 1, ..., 0, 0, 0])

In [186]:
from sklearn.metrics import accuracy_score
accuracy_score(respostas_reais,previsao_final)

0.9976